# FinanceBench: Evaluation Playground


<hr style="border-bottom:0.1px solid gray">

##### (1) API Requirements
Add the following API keys into your `.env` file:

```ruby
OPENAI_API_KEY = 'INSERT API KEY HERE'
ANTHROPIC_API_KEY = 'INSERT API KEY HERE'
REPLICATE_API_TOKEN = 'INSERT API KEY HERE'
```

##### (2) Required Folder Structure

```bash
|-- /
|    |-- data/
|    |      | -- financebench_open_source.jsonl
     |      | -- financebench_document_information.jsonl
|    |-- pdfs/
|           | -- <... provided filings as PDF documents ...>
|    |-- results/
|    |-- vectorstores/
|    |-- evaluation_playground.ipynb
```


<br>
<hr style="border-bottom:0.1px solid gray">

In [1]:
import os
import sys
import json
import pickle
import datetime
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv

parent_dir = os.path.abspath(os.path.join(os.getcwd(), '../'))
sys.path.insert(0, parent_dir)
#LangChain Stuff
# from langchain.document_loaders import PyMuPDFLoader
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain.vectorstores import Chroma
# from langchain.embeddings import OpenAIEmbeddings
# from langchain.chains import RetrievalQA

# LangChain Model Wrappers
# from langchain.chat_models import ChatOpenAI
# from langchain.chat_models import ChatAnthropic
# from langchain.llms.replicate import Replicate
import reynard_react
from dsrag.knowledge_base import KnowledgeBase

# Model Providers
import openai
# import anthropic
# import replicate
# import tiktoken

load_dotenv(dotenv_path="../../.env")

# import ANTHROPIC TOKENIZER
# CLIENT = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
# anthropic_tokenizer = CLIENT.get_tokenizer()
openai.api_key = os.environ['OPENAI_API_KEY']

c:\Users\limyo\anaconda3\envs\dsrag\lib\site-packages\PyPDF2\__init__.py:21: DeprecationWarning: PyPDF2 is deprecated. Please move to the pypdf library instead.
  warnings.warn(


[]


In [2]:
##############################################################################
# MODEL CONFIGS
##############################################################################
# configs = [
#             {"provider": "openai",     "model_name":"gpt-4o-2024-05-13",   "eval_mode":"singleStore",        "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4o-2024-05-13",   "eval_mode":"sharedStore",        "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4o-2024-05-13",   "eval_mode":"inContext",          "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4o-2024-05-13",   "eval_mode":"inContext_reverse",  "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4o-2024-05-13",   "eval_mode":"oracle",             "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4o-2024-05-13",   "eval_mode":"oracle_reverse",     "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4o-2024-05-13",   "eval_mode":"sharedStore",        "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4-1106-preview",  "eval_mode":"sharedStore",        "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4-1106-preview",  "eval_mode":"singleStore",        "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4-1106-preview",  "eval_mode":"inContext",          "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4-1106-preview",  "eval_mode":"closedBook",         "temp":0.01,   "max_tokens":2048},
#             {"provider": "anthropic",  "model_name":"claude-2",            "eval_mode":"inContext",          "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4-1106-preview",  "eval_mode":"oracle",             "temp":0.01,   "max_tokens":2048},
#             {"provider": "replicate",  "model_name":"llama2",              "eval_mode":"sharedStore",        "temp":0.01,   "max_tokens":2048},
#             {"provider": "replicate",  "model_name":"llama2",              "eval_mode":"singleStore",        "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4",               "eval_mode":"sharedStore",        "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4",               "eval_mode":"singleStore",        "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4",               "eval_mode":"closedBook",         "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4",               "eval_mode":"oracle",             "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4-1106-preview",  "eval_mode":"oracle_reverse",     "temp":0.01,   "max_tokens":2048},
#             {"provider": "anthropic",  "model_name":"claude-2",            "eval_mode":"oracle_reverse",     "temp":0.01,   "max_tokens":2048},
#             {"provider": "openai",     "model_name":"gpt-4-1106-preview",  "eval_mode":"inContext_reverse",  "temp":0.01,   "max_tokens":2048},
#             {"provider": "anthropic",  "model_name":"claude-2",            "eval_mode":"inContext_reverse",  "temp":0.01,   "max_tokens":2048},
#             {"provider": "",           "model_name":"",                    "eval_mode":"singleStore",        "temp":None,   "max_tokens":None},       # SPECIAL MODE --> RETRIEVAL ONLY MODE (SINGLE STORE)
#             {"provider": "",           "model_name":"",                    "eval_mode":"sharedStore",        "temp":None,   "max_tokens":None},       # SPECIAL MODE --> RETRIEVAL ONLY MODE (SHARED STORE)
# ]

# replicate_model_mapping = dict({
#             "llama2": "meta/llama-2-70b-chat:02e509c789964a7ea8736978a43525956ef40397be9033abf9fd2badfe68c9e3"
#         })

##############################################################################
# DATASET CONFIG
##############################################################################
PATH_CURRENT = os.path.abspath(os.getcwd())
print(PATH_CURRENT)
PATH_DATASET_JSONL = PATH_CURRENT + "/custom_data/financebench_open_source.jsonl"
PATH_DOCUMENT_INFO_JSONL = PATH_CURRENT + "/custom_data/financebench_document_information.jsonl"
PATH_RESULTS = PATH_CURRENT + "/results/"
PATH_PDFS = PATH_CURRENT + "/custom_pdfs/"

# Choose DATASET PORTION:
# - ALL: Full Dataset
# - OPEN_SOURCE: Open Source Part (n=150)
# - CLOSED_SOURCE: Closed Source Part --> Request access at contact@patronus.ai
DATASET_PORTION = "OPEN_SOURCE"   

##############################################################################
# VECTOR STORE SETUP
##############################################################################
# VS_CHUNK_SIZE = 1024
# VS_CHUNK_OVERLAP = 30
# VS_DIR_VS = PATH_CURRENT + "/vectorstores"

c:\SUTD_STUFF\Computing\Capstone\github\AI-Agents-For-ST\flask\benchmark


In [3]:
##############################################################################
# LOAD DATASET
##############################################################################

# Load Full Dataset 
print(PATH_DATASET_JSONL)
if os.path.exists(PATH_DATASET_JSONL):
    try:
        # Load the dataset
        df_questions = pd.read_json(PATH_DATASET_JSONL, lines=True)
        print("Data loaded successfully")
    except ValueError as e:
        print(f"Error loading JSON: {e}")
else:
    print("File does not exist")

if os.path.exists(PATH_DOCUMENT_INFO_JSONL):
    try:
        # Load the dataset
        df_meta = pd.read_json(PATH_DOCUMENT_INFO_JSONL, lines=True)
        print("Data loaded successfully")
    except ValueError as e:
        print(f"Error loading JSON: {e}")
else:
    print("File does not exist")

# df_questions = pd.read_json(PATH_DATASET_JSONL, lines=True)
# df_meta = pd.read_json(PATH_DOCUMENT_INFO_JSONL, lines=True)
df_full = pd.merge(df_questions, df_meta, on="doc_name")

# Get all docs
df_questions = df_questions.sort_values('doc_name')
ALL_DOCS = df_questions['doc_name'].unique().tolist()
print(f"Total number of distinct PDF: {len(ALL_DOCS)}")

# Select relevant dataset portion
if DATASET_PORTION != "ALL":
    df_questions = df_questions.loc[df_questions["dataset_subset_label"]==DATASET_PORTION]
# df_questions = df_questions.loc[df_questions["dataset_subset_label"]]
print(f"Number of questions: {len(df_questions)}")

# Check relevant documents
df_questions = df_questions.sort_values('doc_name')
docs = df_questions['doc_name'].unique().tolist()
print(f"Number of distinct PDF: {len(docs)}")

c:\SUTD_STUFF\Computing\Capstone\github\AI-Agents-For-ST\flask\benchmark/custom_data/financebench_open_source.jsonl
Data loaded successfully
Data loaded successfully
Total number of distinct PDF: 64
Number of questions: 114
Number of distinct PDF: 64


In [4]:
question_list = []
for i in range(len(df_questions)):
    question_list.append((df_questions.iloc[i]["question"], df_questions.iloc[i]["answer"]))

results = []

In [131]:
# exception_ls = []
# for i, qa in enumerate(question_list):
#     try:
#         res = reynard_react.response(qa[0], llm_name=0, reranker=0, use_graph = 0)
#         output = ''.join([i for i in res])
#         results.append({
#                             "question" : qa[0],
#                             "gold_answer": qa[1],
#                             "model_response": output,
#                         })
#         print(f"question {i} completed.")
#     except:
#         print(f"question {i} exceeded rate limit.")
#         exception_ls.append(i)
# print("Creating csv file")

index = 114

res = reynard_react.response(question_list[index], llm_name=0, reranker=0, use_graph = 0)
output = ''.join([i for i in res])
results.append({
    "question" : question_list[index][0],
    "gold_answer": question_list[index][1],
    "model_response": output,
                        })

IndexError: list index out of range

In [132]:
print(len(results))

101


In [135]:
# from dsrag.knowledge_base import KnowledgeBase
# from dsrag.reranker import CohereReranker, NoReranker
# from dsrag.database.vector.chroma_db import ChromaDB
# import chromadb
# import os
# chromadb.api.client.SharedSystemClient.clear_system_cache()

df_results = pd.DataFrame(results)
df_results.to_csv(PATH_CURRENT+"/model_results.csv")

# STORAGE_DIR = "\\storage"
# current_path = os.getcwd()
# full_path = current_path+STORAGE_DIR
# print(full_path)

# # db = ChromaDB("uss_kb_id", full_path)

# kb = KnowledgeBase("uss_kb_id", reranker=CohereReranker(), vector_db=ChromaDB("uss_kb_id", full_path), storage_directory=full_path)

In [134]:
df_results

,question,gold_answer,model_response
0,What is the FY2018 capital expenditure amount ...,$1577.00,The FY2018 capital expenditure amount for 3M i...
1,Assume that you are a public equities analyst....,$8.70,$8.70 billion
2,Is 3M a capital-intensive business based on FY...,"No, the company is managing its CAPEX and Fixe...","Based on FY2022 data, 3M is not considered a c..."
3,What drove operating margin change as of FY202...,Operating Margin for 3M in FY2022 has decrease...,The operating margin for 3M in FY2022 decrease...
4,"If we exclude the impact of M&A, which segment...",The consumer segment shrunk by 0.9% organically.,The consumer segment has dragged down 3M's ove...
...,...,...,...
96,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,Yes. Verizon's capital intensity ratio was app...
97,Has Verizon increased its debt on balance shee...,No. Verizon's debt decreased by $229 million.,No. Verizon's debt decreased by $229 million b...
98,What is FY2018 days payable outstanding (DPO) ...,42.69,To calculate the FY2018 days payable outstandi...
99,Based on the information provided primarily in...,0.2%,The FY2018 - FY2019 change in unadjusted opera...


In [2]:
import pandas as pd
# ! pip install openai==0.28
# original openai==1.52.2
import openai  # Ensure you're using the latest OpenAI library

# Load the CSV file
data = pd.read_csv('model_results.csv')

def evaluate_response(gold_answer, model_response):
    prompt = (
        f"Evaluate the following model response against the gold answer. "
        f"Be lenient with minor discrepancies such as rounding differences and focus on the overall correctness. As well as sentence structure, as long as their meaning are roughly the same.\n\n"
        f"Gold Answer: {gold_answer}\n"
        f"Model Response: {model_response}\n\n"
        f"Do they match? Please provide a score of 0 or 1, "
        f"where 1 means a passing model response and 0 that the model response fails. "
        f"Explain your reasoning briefly."
    )

    response = openai.ChatCompletion.create(
        model='gpt-4o-mini',  # Ensure you're using a supported model
        messages=[
            {"role": "system", "content": "You are an evaluator tasked with scoring model responses."},
            {"role": "user", "content": prompt}
        ]
    )
    
    # Extract the content of the response
    return response.choices[0].message['content']

# Evaluate each response
results = []
for index, row in data.iterrows():
    gold_answer = row['gold_answer']
    model_response = row['model_response']
    
    evaluation = evaluate_response(gold_answer, model_response)
    results.append({
        'question': row['question'],
        'gold_answer': gold_answer,
        'model_response': model_response,
        'evaluation': evaluation
    })

# Convert results to DataFrame and save if needed
results_df = pd.DataFrame(results)
results_df.to_csv('evaluation_results.csv', index=False)

print("Evaluation completed and results saved.")

Evaluation completed and results saved.


In [3]:
! pip install openai==1.52.2

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
  Attempting uninstall: openai
    Found existing installation: openai 0.28.0
    Uninstalling openai-0.28.0:
      Successfully uninstalled openai-0.28.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dsrag 0.3.5 requires aiohttp==3.9.5, but you have aiohttp 3.11.10 which is incompatible.
dsrag 0.3.5 requires aiosignal==1.3.1, but you have aiosignal 1.2.0 which is incompatible.
dsrag 0.3.5 requires annotated-types==0.7.0, but you have annotated-types 0.6.0 which is incompatible.
dsrag 0.3.5 requires exceptiongroup==1.2.1, but you have exceptiongroup 1.2.0 which is incompatible.
dsrag 0.3.5 requires httpcore==1.0.5, but you have httpcore 1.0.2 which is incompatible.
dsrag 0.3.5 requires jsonpointer==3.0.0, but you have jsonpointer 2.1 which is incompatible.
dsrag 0.3.5 requires langchain-core==0.2.12, but you have langchain-core 0.3.36 which is incompatible.
dsrag 0.3.5 requires langchain-text-splitters==0.2.2, but you have langchain-text-splitters 0.3.6 which is incompatible.
dsrag 0.3.5 requires langsmith==0.1

In [10]:
# print(type(results))
# print(results)

In [11]:
##############################################################################
# HELPER FUNCTIONS (PDF-PARSING + VECTOR-STORE SETUPS)
##############################################################################
# def get_pdf_text(doc):
    
#     path_doc = f"{PATH_PDFS}/{doc}.pdf"
#     pdf_reader = PyMuPDFLoader(path_doc)
#     pdf_text = pdf_reader.load()

#     return pdf_text

# def build_vectorstore_retriever(docs, embeddings = OpenAIEmbeddings()):

#     if docs == "all":
#         docs = ALL_DOCS
#         db_path = VS_DIR_VS + "/shared"
#     else:
#         docs = [docs]
#         db_path = VS_DIR_VS + "/" + docs[0]
    
#     # Create Vector Store if not already existing
#     if not os.path.exists(db_path):
        
#         # Create folder for vector store
#         os.mkdir(db_path) 

#         # Create vector store itself --> chrom.sqlite3 database
#         if not os.path.exists(f"{db_path}/chroma.sqlite3"):
#             vectordb = Chroma(persist_directory=db_path, embedding_function=embeddings)
#             vectordb.persist()
    
#             # Add Documents to Vector store    
#             for doc in docs:
#                 pdf_text = get_pdf_text(doc)
#                 text_splitter = RecursiveCharacterTextSplitter(
#                     chunk_size = VS_CHUNK_SIZE,
#                     chunk_overlap = VS_CHUNK_OVERLAP,
#                 )
#                 splitted_texts = text_splitter.split_documents(pdf_text)
        
#                 # Add to vector store
#                 vectordb.add_documents(documents=splitted_texts)
#                 vectordb.persist()

#     else:
#         vectordb = Chroma(persist_directory=db_path, embedding_function=embeddings)

#     return vectordb.as_retriever(), vectordb

##############################################################################
# MODEL + CALL HANDLERS
##############################################################################

# def get_max_context_length(prompt, anthropic_cutoff=95000, openai_cutoff=105000):

#     # (0) Check Anthropic Tokenizer
#     tokens_anthropic = anthropic_tokenizer.encode(prompt)
#     nb_tokens_anthropic = len(tokens_anthropic)
#     number_of_chars_anthropic = len(prompt)
    
#     if nb_tokens_anthropic > anthropic_cutoff:
#         tokens_anthropic_tokens = tokens_anthropic.tokens
#         token_lengths_anthropic = [len(token) for token in tokens_anthropic_tokens]
#         number_of_chars_anthropic = sum(token_lengths_anthropic[:anthropic_cutoff])
        

#     # (1) Check OpenAI Tokenizer
#     tokenizer_openai = tiktoken.encoding_for_model("gpt-4-1106-preview")
#     tokens_openai = tokenizer_openai.encode(prompt)
#     nb_tokens_openai = len(tokens_openai)
#     number_of_chars_openai = len(prompt)

#     if nb_tokens_openai > openai_cutoff:
#         tokens_openai_tokens = [tokenizer_openai.decode_single_token_bytes(token) for token in tokens_openai]
#         token_lengths_openai = [len(token) for token in tokens_openai_tokens]
#         number_of_chars_openai = sum(token_lengths_openai[:openai_cutoff])

#     # Cut prompt depending on minimal length limit
#     number_of_chars = min(number_of_chars_openai, number_of_chars_anthropic)

#     return number_of_chars

# def get_model(provider="openai", model_name="gpt-4o-mini", temp=0.01, max_tokens=2048):

#     if provider == "openai":
#         return ChatOpenAI(
#             model_name=model_name, 
#             temperature=temp, 
#             max_tokens=max_tokens
#             )
        
#     elif provider == "anthropic":
#         return ChatAnthropic(
#             model=model_name,
#             temperature=temp, 
#             max_tokens_to_sample=max_tokens, 
#             anthropic_api_key=os.environ['ANTHROPIC_API_KEY']
#             )
    
#     elif provider == "replicate":
#         if model_name in replicate_model_mapping:
#             return Replicate(
#                 model=replicate_model_mapping[model_name],
#                 model_kwargs={
#                     'temperature': temp, 
#                     'max_new_tokens': max_tokens
#                     },
#             )
#         else:
#             raise ValueError("Unknown Model")
        
#     else:
#         return None


# def get_answer(model, eval_mode, question, context, retriever, retriever_only=False):

#     retrieved_documents = []

#     if eval_mode == "closedBook":
#         prompt = f"Answer this question: {question}"
#         answer = model.predict(prompt)
        
#     elif eval_mode == "oracle":
#         prompt = f"Answer this question: {question} \nHere is the relevant evidence that you need to answer the question:\n[START OF FILING] {context} [END OF FILING]"
#         answer = model.predict(prompt)

#     elif eval_mode == "oracle_reverse":
        
#         prompt = f"Context:\n[START OF FILING] {context} [END OF FILING\n\n Answer this question: {question} \n"
#         answer = model.predict(prompt)

#     elif eval_mode in ["inContext",  "inContext_reverse"]:
        
#         # Context Cutoff to satisfy max tokens
#         max_number_of_chars = get_max_context_length(context)
#         context = context[:max_number_of_chars]
        
#         if eval_mode == "inContext":
#             prompt = f"Answer this question: {question} \nHere is the relevant filing that you need to answer the question:\n[START OF FILING] {context} [END OF FILING]"
#         else:
#             prompt = f"Context:\n[START OF FILING] {context} [END OF FILING]\n\n Answer this question: {question}\n"

#         answer = model.predict(prompt)

#     elif eval_mode == "singleStore" or eval_mode == "sharedStore":
        
#         # Retrieval-only mode if model=None (No LLM calls, only queries in VectorDB)
#         if not model:           
#             prompt = f"{question}"
#             s = retriever.invoke(prompt)
#             return ("", s)

#         else:

#             # Don't add a question prefix as RetrievalQA will do some automatic prompt wrapping
#             # --> This can replace by more advanced Retrieval Strategies
#             prompt = f"{question}"
#             qa = RetrievalQA.from_chain_type(
#                 llm=model,
#                 chain_type="stuff",
#                 retriever=retriever,
#                 return_source_documents=True,
#             )
#             s = qa(prompt)
            
#             answer = s["result"]
#             retrieved_documents = s["source_documents"]


    
#     return (answer, retrieved_documents)


In [12]:
##############################################################################
# EVALUATION
##############################################################################

# Specify evaluation model
# model_config = configs[0]

# Set evaluation questions
# df_eval = df_questions


# Get the model
# model = get_model(provider=model_config["provider"],
#                   model_name=model_config["model_name"],
#                   temp=model_config["temp"],
#                   max_tokens=model_config["max_tokens"])

# print(f"--> Evaluating: {model_config['model_name']} / {model_config['eval_mode']}")

# last_docs = None
# results = []

# Run evaluation on the model  --> Sort along doc_name to reuse retriever configs in memory
# for k, (idx, row) in tqdm(enumerate(df_eval.sort_values("doc_name").iterrows()), total=len(df_eval)):
        
    
#     # (A) Setup Context or Retriever
#     if model_config["eval_mode"] == "closedBook":
#         retriever = None
#         context = ""
    
#     elif model_config["eval_mode"] in ["inContext", "inContext_reverse"]:
#         retriever = None
#         docs = row["doc_name"]
#         if not (last_docs == docs):
#             pages = get_pdf_text(row["doc_name"])
#             context = "\n\n".join([page.page_content for page in pages])
            
    
#     elif model_config["eval_mode"] in ["oracle", "oracle_reverse"]:
#         context = "\n\n".join([evidence["evidence_text_full_page"] for evidence in row["evidence"]])
#         retriever = None

#     elif model_config["eval_mode"] in ["singleStore", "sharedStore"]:
#         context = ""
#         docs = "all"

#         if model_config["eval_mode"] == "singleStore":
#             docs = row["doc_name"]
        
#         if not (last_docs == docs):
#             retriever, _ = build_vectorstore_retriever(docs=docs)
#             last_docs = docs


#     else:
#         raise ValueError("Unknown 'eval_mode'!")


#     # (B) Model Call
#     (answer, retrieved_documents) = get_answer(
#                                         model=model, 
#                                         eval_mode=model_config["eval_mode"], 
#                                         question=row["question"], 
#                                         context=context, 
#                                         retriever=retriever
#                                         )
    

#     # (C) Bookkeeping
#     results.append({
#                         **model_config, 
#                         "financebench_id" : row["financebench_id"],
#                         "question" : row["question"],
#                         "gold_answer": row["answer"],
#                         "model_answer": answer,
#                         "retrieved_documents" : retrieved_documents,
#                     })

# df_results = pd.DataFrame(results)
# df_results.to_csv(PATH_RESULTS + "/" + model_config["model_name"] + "_" + model_config["eval_mode"] + ".csv")